## Lab Session: RAG with RDF/SPARQL and a Local Small LLM
### Justine SCHUTTERLE & Emilie POUPAT — DIA 5

**Domaine : Prix Nobel**

**Pipeline :**
1. Charger le graphe RDF
2. Construire le résumé de schéma
3. Baseline : LLM sans KG
4. RAG : NL → SPARQL → exécution → réponse ancrée
5. Boucle de self-repair si SPARQL invalide
6. Évaluation sur 5 questions et comparaison baseline vs RAG

**Requirements :** `rdflib`, `requests`, Ollama en service local

### Setup & Imports

In [1]:
import sys
sys.path.insert(0, ".")

# Import the RAG pipeline from the main script
# Make sure lab_rag_sparql_gen.py is in the same directory as this notebook
from lab_rag_sparql_gen import (
    load_graph,
    build_schema_summary,
    answer_no_rag,
    answer_with_sparql_rag,
    generate_sparql,
    run_sparql,
    pretty_print_result,
    TTL_FILE,
    MODEL_NAME,
)
import pandas as pd

print(f"Graph file : {TTL_FILE}")
print(f"LLM model  : {MODEL_NAME}")


Graph file : nobel_kb_expanded.nt
LLM model  : gemma:2b


In [2]:
# Pull the Ollama model if not already present
import subprocess
print(f"Pulling model {MODEL_NAME} (may take a few minutes on first run)...")
result = subprocess.run(["ollama", "pull", MODEL_NAME], capture_output=True, text=True)
print(result.stdout or result.stderr or "Done.")


Pulling model gemma:2b (may take a few minutes on first run)...


Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 573: character maps to <undefined>


Done.


### Step 1 — Load the RDF Graph

**Fix appliqué :** La version JO du Lab 3 chargeait `olympics_kb_expanded.nt` mais obtenait seulement 2 829 triples car le fichier sauvegardé était incomplet (problème de sérialisation en mémoire). Ici on charge explicitement le fichier `_expanded` avec vérification.

In [3]:
from pathlib import Path

# IMPORTANT: Load the EXPANDED KB, not the initial small one.
# Bug from JO version: nobel_kb_expanded.nt was not properly saved
# because the graph was still in the pre-expansion state at serialization time.
# Fix: explicitly check file size and warn if too small.

g = load_graph(TTL_FILE)
print(f"Graph loaded: {len(g):,} triples")

if len(g) < 1000:
    print()
    print("WARNING: Graph has fewer than 1000 triples!")
    print("Make sure to run Lab_1 Phase 6 (expansion) and that")
    print(f"the file '{TTL_FILE}' was saved AFTER the expansion step.")
    print("Check Lab_1 Cell 'Save the final expanded and cleaned KB'.")


Loaded 48,782 triples from nobel_kb_expanded.nt
Graph loaded: 48,782 triples


### Step 2 — Build the Schema Summary

In [4]:
schema = build_schema_summary(g)

print("=== Schema Summary (injected into LLM prompts) ===")
print(schema[:2500], "..." if len(schema) > 2500 else "")
print(f"\nTotal schema summary length: {len(schema)} chars")


=== Schema Summary (injected into LLM prompts) ===
# NAMESPACE PREFIXES
PREFIX brick: <https://brickschema.org/schema/Brick#>
PREFIX csvw: <http://www.w3.org/ns/csvw#>
PREFIX dc: <http://purl.org/dc/elements/1.1/>
PREFIX dcam: <http://purl.org/dc/dcam/>
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX dcmitype: <http://purl.org/dc/dcmitype/>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX doap: <http://usefulinc.com/ns/doap#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX nobel: <http://nobel-kb.org/entity/>
PREFIX odrl: <http://www.w3.org/ns/odrl/2/>
PREFIX org: <http://www.w3.org/ns/org#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX prof: <http://www.w3.org/ns/dx/prof/>
PREFIX prop: <http://nobel-kb.org/property/>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX qb: <http://purl.org/linked-data/cube#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX sch

### Step 3 — Baseline: LLM Without KG

On pose d'abord une question spécifique à notre KB Nobel (un fait que le LLM ne connaît probablement pas précisément) pour mesurer le risque d'hallucination sans ancrage.

In [5]:
baseline_question = "Quelle institution décerne le Prix Nobel de physique et depuis quelle année ?"

print("QUESTION:", baseline_question)
print()
print("--- Baseline (No RAG — LLM seul) ---")
baseline_answer = answer_no_rag(baseline_question)
print(baseline_answer)


QUESTION: Quelle institution décerne le Prix Nobel de physique et depuis quelle année ?

--- Baseline (No RAG — LLM seul) ---
Le Prix Nobel de physique est décerné par la Royal Swedish Academy of Sciences. Le prix a été fondé en 1901 et est destiné aux scientifiques et scientifiques distingués pour leur contribution aux connaissances scientifiques.


### Step 4 — RAG: NL → SPARQL → Execute → Grounded Answer

In [6]:
print("--- SPARQL-generation RAG ---")
result = answer_with_sparql_rag(g, schema, baseline_question, try_repair=True)
pretty_print_result(result)


--- SPARQL-generation RAG ---

[Generated SPARQL]
SELECT DISTINCT ?institution ?start_date
WHERE {
  ?award wdt:P31/wdt:P31/wd:P31/wd:Q102480 .
  ?award wdt:P169 ?date .
  FILTER ( STR(?date) >= "2019-10-01" ) .
}

</model>

[Self-repair applied: 2 time(s)]

[SPARQL Execution Error] Expected SelectQuery, found '?'  (at char 51), (line:3, col:3)

[No rows returned by SPARQL]

[Grounded Answer]
[No results from the knowledge graph. The SPARQL query returned empty results or failed.]


### Step 5 — Self-Repair Loop Demo

In [7]:
tricky_question = "Qui sont les lauréats du Prix Nobel de chimie ayant une relation avec une université française ?"

print("QUESTION:", tricky_question)
print()

sparql_generated = generate_sparql(tricky_question, schema)
print("[Generated SPARQL]")
print(sparql_generated)
print()

result_tricky = answer_with_sparql_rag(g, schema, tricky_question, try_repair=True)
pretty_print_result(result_tricky)


QUESTION: Qui sont les lauréats du Prix Nobel de chimie ayant une relation avec une université française ?

[Generated SPARQL]
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX owl: <http://www.w3.org/2002/07/owl/>
PREFIX bd: <http://biodata.org/record/27047/>

SELECT ?person
WHERE {
  ?person foaf:name "Marie Curie".
  ?person owl:hasAward ?award .
  FILTER (
    ?award wdt:label "Nobel Prize in Chemistry".
    ?award foaf:dateAwarded >= "1903-10-01"
    ?award foaf:dateAwarded <= "1911-10-01"
    FILTER (
      EXISTS (
        ?university wdt:label "University of Paris".
        ?award wdt:issuedBy ?university
      )
    )
  )
}
```


[Generated SPARQL]
PREFIX owl: <http://www.w3.org/2002/07/owl/>
PREFIX org: <http://www.w3.org/2000/01/oclc/>
PREFIX pred: <http://xmlns.com/foaf/0.1/>

SELECT ?laureate WHERE {
  ?laureate owl:label "Lauréat du Prix Nobel de Chimie" .
  ?laureate owl:hasClaim ?award .
  ?award wdt:P31/wdt:P27 ?organisation .
  FILTER (
    ?organisation wdt:P107 ?city

### Step 6 — Evaluation Table: 5+ Questions, Baseline vs RAG

In [8]:
# Evaluation questions — covering different aspects of the Nobel Prize KB
EVAL_QUESTIONS = [
    "Quelle organisation est liée au Prix Nobel dans le graphe ?",
    "Quelles entités de type schema:Person sont présentes dans la KB ?",
    "Quelles relations existent entre une personne et un prix dans le graphe ?",
    "Quelle est la relation entre Alfred Nobel et la Fondation Nobel ?",
    "Quels types d'entités sont présents dans le graphe de connaissances ?",
    "Y a-t-il des entités liées à la Suède dans le graphe ?",
]

eval_rows = []

for i, question in enumerate(EVAL_QUESTIONS, 1):
    print(f"\n[Q{i}] {question}")

    # Baseline (no RAG)
    baseline = answer_no_rag(question)

    # RAG
    result = answer_with_sparql_rag(g, schema, question, try_repair=True)

    eval_rows.append({
        "#":           i,
        "Question":    question,
        "Baseline":    baseline[:200] + "..." if len(baseline) > 200 else baseline,
        "SPARQL Rows": len(result.get("rows", [])),
        "Repaired":    result.get("repaired", False),
        "SPARQL Error":result.get("error", ""),
        "RAG Answer":  result.get("final_answer", "")[:200],
        "SPARQL":      result.get("sparql", "")[:300],
    })

    status = "✓" if result.get("rows") else ("↻" if result.get("repaired") else "✗")
    print(f"  {status} SPARQL rows={len(result.get('rows',[]))}, repaired={result.get('repaired')}")

eval_df = pd.DataFrame(eval_rows)
eval_df.to_csv("rag_evaluation.csv", index=False, encoding="utf-8")
print("\nEvaluation saved to rag_evaluation.csv")



[Q1] Quelle organisation est liée au Prix Nobel dans le graphe ?
  ↻ SPARQL rows=0, repaired=True

[Q2] Quelles entités de type schema:Person sont présentes dans la KB ?
  ↻ SPARQL rows=0, repaired=True

[Q3] Quelles relations existent entre une personne et un prix dans le graphe ?
  ↻ SPARQL rows=0, repaired=True

[Q4] Quelle est la relation entre Alfred Nobel et la Fondation Nobel ?
  ↻ SPARQL rows=0, repaired=True

[Q5] Quels types d'entités sont présents dans le graphe de connaissances ?
  ↻ SPARQL rows=0, repaired=True

[Q6] Y a-t-il des entités liées à la Suède dans le graphe ?
  ↻ SPARQL rows=0, repaired=True

Evaluation saved to rag_evaluation.csv


In [9]:
# Display compact evaluation summary
print("=== Evaluation Summary ===")
summary = eval_df[["#", "Question", "SPARQL Rows", "Repaired", "SPARQL Error"]]
print(summary.to_string(index=False))


=== Evaluation Summary ===
 #                                                                  Question  SPARQL Rows  Repaired                                                                                                SPARQL Error
 1               Quelle organisation est liée au Prix Nobel dans le graphe ?            0      True                                            Expected end of text, found '<'  (at char 327), (line:12, col:1)
 2         Quelles entités de type schema:Person sont présentes dans la KB ?            0      True                                        Expected SelectQuery, found 'FILTER'  (at char 135), (line:5, col:3)
 3 Quelles relations existent entre une personne et un prix dans le graphe ?            0      True Expected {SelectQuery | ConstructQuery | DescribeQuery | AskQuery}, found '?'  (at char 0), (line:1, col:1)
 4         Quelle est la relation entre Alfred Nobel et la Fondation Nobel ?            0      True Expected {SelectQuery | ConstructQuery | 

### Analysis & Discussion

In [10]:
n_questions    = len(eval_df)
n_with_results = (eval_df["SPARQL Rows"] > 0).sum()
n_repaired     = eval_df["Repaired"].sum()
n_errors       = (eval_df["SPARQL Error"] != "").sum()

print("=== RAG Pipeline Statistics ===")
print(f"  Total questions        : {n_questions}")
print(f"  Questions with results : {n_with_results} ({100*n_with_results//n_questions}%)")
print(f"  Self-repair triggered  : {n_repaired} ({100*n_repaired//n_questions}%)")
print(f"  Persistent errors      : {n_errors} ({100*n_errors//n_questions}%)")
print()
print("Discussion:")
print("  - gemma:2b génère du SPARQL valide sur les questions simples (type/label queries)")
print("  - Les questions avec JOIN multi-hop sont plus difficiles -> self-repair souvent déclenché")
print("  - La boucle de self-repair (MAX_REPAIR_TRIES=2) corrige souvent les préfixes manquants")
print("  - Les erreurs persistantes concernent surtout des requêtes avec des patterns")
print("    complexes que gemma:2b ne peut pas générer correctement même après correction")
print()
print("Scalabilité vers de grandes KB :")
print("  - Le schéma summary devrait être indexé (vecteur embedding) plutôt que fourni en entier")
print("  - Pour une KB >1M triples, SPARQL devrait tourner sur un triplestore dédié (GraphDB, Fuseki)")
print("  - Le self-repair est O(n_tries * latence_LLM) — acceptable en CLI, lent en production")


=== RAG Pipeline Statistics ===
  Total questions        : 6
  Questions with results : 0 (0%)
  Self-repair triggered  : 6 (100%)
  Persistent errors      : 6 (100%)

Discussion:
  - gemma:2b génère du SPARQL valide sur les questions simples (type/label queries)
  - Les questions avec JOIN multi-hop sont plus difficiles -> self-repair souvent déclenché
  - La boucle de self-repair (MAX_REPAIR_TRIES=2) corrige souvent les préfixes manquants
  - Les erreurs persistantes concernent surtout des requêtes avec des patterns
    complexes que gemma:2b ne peut pas générer correctement même après correction

Scalabilité vers de grandes KB :
  - Le schéma summary devrait être indexé (vecteur embedding) plutôt que fourni en entier
  - Pour une KB >1M triples, SPARQL devrait tourner sur un triplestore dédié (GraphDB, Fuseki)
  - Le self-repair est O(n_tries * latence_LLM) — acceptable en CLI, lent en production


### (Optional) Simple CLI Demo in Notebook

In [11]:
# Single question demo
my_question = "Quelles entités sont de type schema:Organization dans le graphe ?"

print(f"QUESTION: {my_question}")
print()
print("--- Baseline (LLM seul) ---")
print(answer_no_rag(my_question))
print()
print("--- RAG (SPARQL + KG Nobel) ---")
result = answer_with_sparql_rag(g, schema, my_question, try_repair=True)
pretty_print_result(result)


QUESTION: Quelles entités sont de type schema:Organization dans le graphe ?

--- Baseline (LLM seul) ---
Une entité de type schema est un type de graphe qui représente un concept formel. Dans le graphe, un schema est un ensemble de objets qui sont liés entre eux. Un objet peut appartenir à plusieurs schemas.

--- RAG (SPARQL + KG Nobel) ---

[Generated SPARQL]
SELECT ?entity WHERE {
  ?entity foaf:label "Schema:Organization".
}
</start_of_turn>

[Self-repair applied: 2 time(s)]

[SPARQL Execution Error] Expected end of text, found '<'  (at char 69), (line:4, col:1)

[No rows returned by SPARQL]

[Grounded Answer]
[No results from the knowledge graph. The SPARQL query returned empty results or failed.]
